# 🎵 Song-to-Music Generator
Paste any song with `[Verse]`/`[Chorus]`/`[Bridge]` headers (or just blank-line-separated stanzas) and hear it played back in Indian classical style.


In [1]:
# !pip install textblob numpy
# !python -m textblob.download_corpora

import re
import numpy as np
from textblob import TextBlob
from IPython.display import Audio, display

# ── 1. RAGA DEFINITIONS ──────────────────────────────────────────
RAGAS = {
    'Bilawal':      {'notes': ['Sa','Re','Ga','Ma','Pa','Dha','Ni','Sa2'], 'mood': 'happy',   'tempo': 1.00, 'vadi': 'Ga'},
    'Desh':         {'notes': ['Sa','Re','Ga','Ma','Pa','Ni','Sa2'],       'mood': 'happy',   'tempo': 0.90, 'vadi': 'Re'},
    'Yaman':        {'notes': ['Sa','Re','Ga','ma','Pa','Dha','Ni','Sa2'], 'mood': 'peaceful','tempo': 0.80, 'vadi': 'Ga'},
    'Bhairav':      {'notes': ['Sa','re','Ga','Ma','Pa','dha','Ni','Sa2'], 'mood': 'peaceful','tempo': 0.70, 'vadi': 'Ma'},
    'Shivaranjani': {'notes': ['Sa','Re','ga','Pa','Dha','Sa2'],           'mood': 'sad',     'tempo': 0.85, 'vadi': 'ga'},
    'Bhairavi':     {'notes': ['Sa','re','ga','Ma','Pa','dha','ni','Sa2'], 'mood': 'sad',     'tempo': 0.75, 'vadi': 'Sa'},
}

FREQUENCIES = {
    'Sa': 261.63, 're': 277.18, 'Re': 293.66, 'ga': 311.13, 'Ga': 329.63,
    'Ma': 349.23, 'ma': 369.99, 'Pa': 392.00, 'dha': 415.30, 'Dha': 440.00,
    'ni': 466.16, 'Ni': 493.88, 'Sa2': 523.25,
}

SAMPLE_RATE = 44100
print('✅ Config loaded')

✅ Config loaded


In [2]:
# ── 2. SONG STRUCTURE PARSER ─────────────────────────────────────
SECTION_RE = re.compile(r'^\[?(verse|chorus|bridge|pre-chorus|outro|intro|hook|refrain)\s*\d*\]?\s*$', re.IGNORECASE)

def parse_song(raw):
    sections, cur_label, cur_lines, auto_idx = [], None, [], 0
    auto_labels = ['Verse','Chorus','Verse','Chorus','Bridge','Chorus','Outro']
    for line in raw.strip().splitlines():
        line = line.strip()
        if SECTION_RE.match(line) or (line.endswith(':') and len(line) < 20):
            if cur_lines:
                sections.append({'label': cur_label or 'Intro', 'text': ' '.join(cur_lines)})
            cur_label, cur_lines = line.strip('[]:\t '), []
        elif line == '':
            if cur_lines:
                sections.append({'label': cur_label or auto_labels[auto_idx % len(auto_labels)], 'text': ' '.join(cur_lines)})
                cur_label, cur_lines = None, []
                auto_idx += 1
        else:
            cur_lines.append(line)
    if cur_lines:
        sections.append({'label': cur_label or auto_labels[auto_idx % len(auto_labels)], 'text': ' '.join(cur_lines)})
    return sections or [{'label': 'Song', 'text': raw}]


# ── 3. SENTIMENT → RAGA ──────────────────────────────────────────
def pick_raga(text, label=''):
    blob = TextBlob(text)
    p, s = blob.sentiment.polarity, blob.sentiment.subjectivity
    lbl  = label.lower()
    if p > 0.25:
        raga = 'Desh' if 'chorus' in lbl else 'Bilawal'
    elif p < -0.15:
        raga = 'Bhairavi' if 'bridge' in lbl else 'Shivaranjani'
    else:
        raga = 'Bhairav' if s > 0.5 else 'Yaman'
    return raga, RAGAS[raga]


# ── 4. SYLLABLE DURATION ─────────────────────────────────────────
def syllables(word):
    w, count, prev = word.lower().strip(".,!?;:'\""), 0, False
    for ch in w:
        v = ch in 'aeiouy'
        if v and not prev: count += 1
        prev = v
    if w.endswith('e') and count > 1: count -= 1
    return max(1, count)

def note_duration(word, tempo):
    s = syllables(word)
    return round(min(0.25 + (s - 1) * 0.18, 0.9) * tempo, 3)

print('✅ Parser & helpers loaded')

✅ Parser & helpers loaded


In [3]:
# ── 5. MELODIC SEQUENCE BUILDER ──────────────────────────────────
def build_sequence(section):
    raga_name, raga = pick_raga(section['text'], section['label'])
    notes   = raga['notes']
    tempo   = raga['tempo']
    vadi_i  = notes.index(raga['vadi']) if raga['vadi'] in notes else len(notes) // 2
    print(f"  [{section['label']}]  raga: {raga_name}  mood: {raga['mood']}")
    words, seq, idx = section['text'].split(), [], vadi_i
    maxi = len(notes) - 1
    for i, word in enumerate(words):
        clean = word.lower().strip(".,!?;:'\"")
        if not clean: continue
        s    = syllables(clean)
        step = 1 if s <= 1 else (2 if s == 2 else 3)
        direction = 1 if clean[0] in 'aeiou' else -1
        if i % 4 == 3: direction = 1 if idx < vadi_i else -1   # gravity toward vadi
        idx += direction * step
        if idx > maxi: idx = maxi - (idx - maxi)
        elif idx < 0:  idx = abs(idx)
        idx  = max(0, min(idx, maxi))
        dur  = note_duration(clean, tempo)
        if i == len(words) - 1: dur = min(dur * 2.5, 2.0)   # last word rings out
        elif i % 8 == 7:        dur *= 1.5                   # phrase end breathe
        seq.append((word, notes[idx], dur))
    return seq


# ── 6. AUDIO ENGINE (ADSR + harmonics + meend) ───────────────────
def adsr(n, sr, a=0.05, d=0.10, sus=0.75, r=0.12):
    a_s, d_s, r_s = int(a*sr), int(d*sr), int(r*sr)
    s_s = max(0, n - a_s - d_s - r_s)
    env = np.concatenate([np.linspace(0,1,a_s), np.linspace(1,sus,d_s),
                          np.full(s_s,sus), np.linspace(sus,0,r_s)])
    return env[:n]

def synth_note(freq, dur, sr=SAMPLE_RATE):
    n = int(dur * sr)
    t = np.arange(n) / sr
    harmonics = [(1, 1.0), (2, 0.35), (3, 0.15)]
    wave = sum(amp * np.sin(2 * np.pi * freq * h * t) for h, amp in harmonics)
    wave /= sum(amp for _, amp in harmonics)
    return wave * adsr(n, sr)

def render(seq, sr=SAMPLE_RATE):
    blocks = [synth_note(FREQUENCIES.get(note, FREQUENCIES['Sa']), dur, sr)
              for _, note, dur in seq]
    audio = np.concatenate(blocks)
    fade  = int(sr * 0.25)
    audio[:fade]  *= np.linspace(0, 1, fade)
    audio[-fade:] *= np.linspace(1, 0, fade)
    return audio


# ── 7. TABLA PERCUSSION ──────────────────────────────────────────
def tabla_track(total_samples, sr=SAMPLE_RATE, bpm=90):
    beat  = int(sr * 60 / bpm)
    track = np.zeros(total_samples)
    t_hit = np.arange(int(0.05 * sr)) / sr
    dha   = 0.50 * np.sin(2*np.pi*150*t_hit) * np.exp(-25*t_hit)
    tin   = 0.25 * np.sin(2*np.pi*500*t_hit) * np.exp(-50*t_hit)
    pos, b = 0, 0
    while pos + len(dha) < total_samples:
        hit = dha if b % 16 in (0,8) else (tin if b % 16 in (4,12) else dha * 0.25)
        end = min(pos + len(hit), total_samples)
        track[pos:end] += hit[:end-pos]
        pos += beat // 4
        b   += 1
    return track

print('✅ Audio engine loaded')

✅ Audio engine loaded


In [4]:
# ── 8. MAIN PIPELINE ─────────────────────────────────────────────
def song_to_music(raw_lyrics, bpm=90, melody_vol=0.80, tabla_vol=0.25, sr=SAMPLE_RATE):
    sections = parse_song(raw_lyrics)
    print(f"\n🎼  {len(sections)} section(s): {[s['label'] for s in sections]}\n")

    full_audio = []
    silence    = np.zeros(int(sr * 0.4))
    all_seq    = []

    for sec in sections:
        seq   = build_sequence(sec)
        audio = render(seq, sr)
        all_seq.append((sec['label'], seq))
        full_audio.extend([audio, silence])

    # Print word → note table
    print(f"\n{'─'*52}")
    print(f"{'SECTION':<13} {'WORD':<16} {'NOTE':<6} {'DUR':>5}")
    print('─'*52)
    for label, seq in all_seq:
        for word, note, dur in seq:
            print(f"{label:<13} {word:<16} {note:<6} {dur:>4.2f}s")
            label = ''
    print('─'*52)

    melody = np.concatenate(full_audio)
    tabla  = tabla_track(len(melody), sr=sr, bpm=bpm)
    mixed  = melody_vol * melody + tabla_vol * tabla
    peak   = np.max(np.abs(mixed))
    if peak > 0: mixed /= peak

    print(f"\n🎵  Duration: {len(mixed)/sr:.1f}s | BPM: {bpm}")
    print("▶  Playing ...\n")
    display(Audio(mixed, rate=sr, autoplay=False))

print('✅ Pipeline ready — run the next cell!')

✅ Pipeline ready — run the next cell!


In [5]:
# ── 🎤 PASTE YOUR SONG HERE ──────────────────────────────────────
song = """
[Verse 1]
In the stillness of the morning
I hear the river calling me
Beneath the ancient banyan shadow
My heart finds what it needs to be

[Chorus]
Oh take me home take me home
Where the mountains meet the sea
Let the wind carry my sorrow
And bring me back to me

[Bridge]
There is no darkness without light
No silence without song
Every parting every sadness
Means somewhere I belong

[Chorus]
Oh take me home take me home
Where the mountains meet the sea
Let the wind carry my sorrow
And bring me back to me
"""

song_to_music(
    raw_lyrics = song,
    bpm        = 82,    # tempo of tabla
    melody_vol = 0.80,  # 0-1
    tabla_vol  = 0.25,  # 0 = no tabla
)


🎼  4 section(s): ['Verse 1', 'Chorus', 'Bridge', 'Chorus']

  [Verse 1]  raga: Yaman  mood: peaceful
  [Chorus]  raga: Yaman  mood: peaceful
  [Bridge]  raga: Bilawal  mood: happy
  [Chorus]  raga: Yaman  mood: peaceful

────────────────────────────────────────────────────
SECTION       WORD             NOTE     DUR
────────────────────────────────────────────────────
Verse 1       In               ma     0.20s
              the              Ga     0.20s
              stillness        Sa     0.34s
              of               Re     0.20s
              the              Sa     0.20s
              morning          Ga     0.34s
              I                ma     0.20s
              hear             Ga     0.30s
              the              Re     0.20s
              river            Re     0.34s
              calling          Re     0.34s
              me               Ga     0.20s
              Beneath          Sa     0.34s
              the              Re     0.20s
            